[![Open in Colab](https://img.shields.io/badge/Open%20in-Colab-orange?logo=googlecolab)](https://colab.research.google.com/github/YOUR_USERNAME/satellite-change-detection/blob/main/notebooks/05_cbam_attention.ipynb)

# 05 CBAM Attention

Add CBAM attention to the decoder skip pathway and retrain the final backbone variant.

**Prerequisites:** Google Colab GPU runtime, LEVIR-CD dataset access, repo files available.

**Expected runtime:** 45-75 minutes on a T4 GPU


## CBAM Decoder Attention

This notebook enables CBAM on each decoder skip tensor so the fusion pathway can emphasize building edges and suppress irrelevant context before concatenation.

In [ ]:
from pathlib import Path
from copy import deepcopy
from src.factory import load_config, build_model

step4_config = deepcopy(load_config(Path('configs/full.yaml')))
step4_config['data']['root'] = '/content/levir_data'
step4_config['model']['use_cbam'] = True
step4_config['paths']['results_dir'] = 'results/step4'
step4_config['paths']['best_checkpoint_name'] = 'siamese_cbam_best.pth'
step4_config['paths']['history_prefix'] = 'step4'
model = build_model(step4_config)
print(model.decoder.skip_attn)
# Expected output: five CBAM blocks attached to the decoder skip pathway.


## CBAM Training Run

This cell retrains the ResNet-based model with CBAM enabled, using the same BCE-Dice loss and staged fine-tuning strategy as Step 3.

In [ ]:
from src.train import train_from_config

step4_summary = train_from_config(step4_config, config_name='step4_resnet18_cbam')
step4_summary['test_metrics']
# Expected output: a modest but meaningful gain over the plain ResNet decoder, ideally reaching or exceeding 0.72 F1.


## Notebook Summary

CBAM is now integrated at each skip fusion point, which should help sharpen small structures and reduce false positives around building boundaries.